# 🚀 7B Model: FP16 vs INT8 Quantization Benchmark

This notebook runs 3 experiments:
1. **FP16** — Raw 14GB model, no quantization
2. **INT8** — bitsandbytes quantization (7GB)
3. **Side-by-side comparison** — VRAM usage, TTFT, throughput

In [ ]:
# Cell 1: Install vLLM and dependencies
import subprocess, sys
print('Installing vLLM...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'vllm==0.6.3', '-q'], check=True)
print('Done!')

In [ ]:
# Cell 2: Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU Info:', result.stdout.strip())
import torch
print(f'PyTorch CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: EXPERIMENT 1 — FP16 (Raw 7B Model, No Quantization)
print('=' * 60)
print('EXPERIMENT 1: Loading Qwen 7B in FP16 (full precision)')
print('Expected VRAM: ~14 GB for weights')
print('=' * 60)

import time, torch
from vllm import LLM, SamplingParams

# Track VRAM before
torch.cuda.empty_cache()
vram_before = torch.cuda.memory_allocated() / 1e9

t_start = time.time()
llm_fp16 = LLM(
    model='Qwen/Qwen2.5-7B-Instruct',
    dtype='float16',
    max_model_len=2048,
    gpu_memory_utilization=0.90,
    enable_prefix_caching=True,
)
load_time_fp16 = time.time() - t_start

vram_after = torch.cuda.memory_allocated() / 1e9
vram_used_fp16 = vram_after - vram_before

print(f'\n✅ FP16 Model Loaded!')
print(f'   Load time : {load_time_fp16:.1f}s')
print(f'   VRAM used : {vram_used_fp16:.2f} GB')
print(f'   VRAM free : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB left for KV cache')

In [ ]:
# Cell 4: FP16 Inference Benchmark
import time
from vllm import SamplingParams

sampling_params = SamplingParams(temperature=0.0, max_tokens=100)

# Test prompts of different lengths
prompts = {
    'short (50 tokens)':  'What is machine learning? Explain in 2 sentences.',
    'medium (200 tokens)': 'Explain the transformer architecture in detail. Cover: 1) Self-attention mechanism, 2) Multi-head attention, 3) Feed-forward layers, 4) Positional encoding, 5) Layer normalization.',
    'long (500 tokens)':   ' '.join(['Explain quantization in neural networks.'] * 15),
}

fp16_results = {}
print('Running FP16 inference benchmarks...\n')

for label, prompt in prompts.items():
    # Warm up
    llm_fp16.generate([prompt], sampling_params)
    
    # Measure TTFT (Time to First Token) via total generation time / tokens
    times = []
    for _ in range(3):
        t0 = time.time()
        output = llm_fp16.generate([prompt], sampling_params)
        t1 = time.time()
        times.append(t1 - t0)
    
    avg_time = sum(times) / len(times)
    tokens_out = len(output[0].outputs[0].token_ids)
    throughput = tokens_out / avg_time
    
    fp16_results[label] = {'latency_ms': avg_time * 1000, 'throughput': throughput}
    print(f'  {label}')
    print(f'    Latency  : {avg_time*1000:.1f} ms')
    print(f'    Throughput: {throughput:.1f} tokens/sec')
    print()

In [ ]:
# Cell 5: Free FP16 model, load INT8
print('Freeing FP16 model from VRAM...')
del llm_fp16
torch.cuda.empty_cache()
import gc; gc.collect()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB used')

print('\n' + '='*60)
print('EXPERIMENT 2: Loading Qwen 7B in INT8 (bitsandbytes quantization)')
print('Expected VRAM: ~7 GB for weights (50% reduction!)')
print('='*60)

from vllm import LLM

vram_before = torch.cuda.memory_allocated() / 1e9
t_start = time.time()

llm_int8 = LLM(
    model='Qwen/Qwen2.5-7B-Instruct',
    quantization='bitsandbytes',
    load_format='bitsandbytes',
    max_model_len=4096,           # More context fits because weights use less VRAM!
    gpu_memory_utilization=0.90,
    enable_prefix_caching=True,
)

load_time_int8 = time.time() - t_start
vram_used_int8 = torch.cuda.memory_allocated() / 1e9 - vram_before

print(f'\n✅ INT8 Model Loaded!')
print(f'   Load time : {load_time_int8:.1f}s')
print(f'   VRAM used : {vram_used_int8:.2f} GB')
print(f'   VRAM free : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB left for KV cache')
print(f'   VRAM saved vs FP16: {vram_used_fp16 - vram_used_int8:.2f} GB ({((vram_used_fp16 - vram_used_int8)/vram_used_fp16)*100:.0f}% reduction)')

In [ ]:
# Cell 6: INT8 Inference Benchmark (same prompts)
int8_results = {}
print('Running INT8 inference benchmarks...\n')

for label, prompt in prompts.items():
    llm_int8.generate([prompt], sampling_params)
    
    times = []
    for _ in range(3):
        t0 = time.time()
        output = llm_int8.generate([prompt], sampling_params)
        t1 = time.time()
        times.append(t1 - t0)
    
    avg_time = sum(times) / len(times)
    tokens_out = len(output[0].outputs[0].token_ids)
    throughput = tokens_out / avg_time
    
    int8_results[label] = {'latency_ms': avg_time * 1000, 'throughput': throughput}
    print(f'  {label}')
    print(f'    Latency  : {avg_time*1000:.1f} ms')
    print(f'    Throughput: {throughput:.1f} tokens/sec')
    print()

In [ ]:
# Cell 7: Final Comparison Table
print('\n' + '='*70)
print('📊 FINAL COMPARISON: FP16 vs INT8')
print('='*70)
print(f'{"Metric":<30} {"FP16":>15} {"INT8":>15} {"Difference":>15}')
print('-'*70)

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'{"Weight VRAM":<30} {vram_used_fp16:>14.1f}G {vram_used_int8:>14.1f}G {((vram_used_int8-vram_used_fp16)/vram_used_fp16)*100:>13.0f}%')
print(f'{"KV Cache Room":<30} {total_vram-vram_used_fp16:>13.1f}G {total_vram-vram_used_int8:>13.1f}G {"INT8 wins":>15}')
print(f'{"Max Context (tokens)":<30} {"2048":>15} {"4096":>15} {"INT8 2x more":>15}')

print()
for label in prompts.keys():
    fp16_lat = fp16_results[label]['latency_ms']
    int8_lat = int8_results[label]['latency_ms']
    fp16_tps = fp16_results[label]['throughput']
    int8_tps = int8_results[label]['throughput']
    diff_lat = ((int8_lat - fp16_lat) / fp16_lat) * 100
    print(f'{label + " latency":<30} {fp16_lat:>13.0f}ms {int8_lat:>13.0f}ms {diff_lat:>+13.0f}%')
    print(f'{label + " throughput":<30} {fp16_tps:>12.1f}t/s {int8_tps:>12.1f}t/s')
    print()

print('='*70)
print('Key Takeaway:')
print('  INT8 saves ~50% VRAM (weights: 14GB → 7GB)')
print('  INT8 latency is usually within 5-10% of FP16')
print('  INT8 allows 2x longer context windows on the same GPU')
print('  → This is WHY quantization was invented!')